In [ ]:
# !pip install zarr
# !pip install imageio
# !pip install rerun-sdk

In [ ]:
import zarr

zarr_path = "../pg3d_xarm7_fixed.zarr/"

root = zarr.open(zarr_path, mode="r")


print(root)


<zarr.hierarchy.Group '/' read-only>


In [2]:
print("\n===== Zarr Tree =====")
print(root.tree())


===== Zarr Tree =====
/
 ├── data
 │   ├── action (345728, 7) float32
 │   ├── eef_pos (345728, 3) float32
 │   ├── goal_pos (345728, 3) float32
 │   ├── goal_relative (345728, 3) float32
 │   ├── point_cloud (345728, 1024, 3) float32
 │   ├── point_valid_mask (345728, 1024) bool
 │   ├── robot_mask (345728, 1024) bool
 │   ├── sim_action (345728, 7) float32
 │   ├── state (345728, 13) float32
 │   ├── success (345728,) bool
 │   ├── target_position (345728, 3) float32
 │   ├── tcp_pose (345728, 7) float32
 │   ├── trajectory_family_id (345728, 1) int64
 │   └── trajectory_family_onehot (345728, 12) float32
 └── meta
     └── episode_ends (5000,) int64


In [4]:
import json
import numpy as np
import zarr

root = zarr.open("../pg3d_xarm7_fixed.zarr", mode="r")

episode_ends = root["meta/episode_ends"][:]
target_pos = root["data/target_position"]

mapping = {}

start = 0

for ep, end in enumerate(episode_ends):
    # Target position at the first frame of the episode
    target = target_pos[start].tolist()

    mapping[ep] = {
        "start_idx": int(start),
        "end_idx": int(end),
        "num_frames": int(end - start),
        "target_position": [float(x) for x in target],
    }

    start = int(end)

with open("episode_target_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

print(f"Saved mapping for {len(mapping)} episodes.")

Saved mapping for 5000 episodes.


In [3]:
print("\n===== Arrays =====")

arrays = []

def collect(name, obj):
    if isinstance(obj, zarr.Array):
        arrays.append((name, obj))

root.visititems(collect)

for name, arr in arrays:
    print(f"\n{name}")
    print(f"  Shape : {arr.shape}")
    print(f"  Dtype : {arr.dtype}")
    print(f"  Chunks: {arr.chunks}")


===== Arrays =====

data/action
  Shape : (345728, 7)
  Dtype : float32
  Chunks: (1024, 7)

data/eef_pos
  Shape : (345728, 3)
  Dtype : float32
  Chunks: (1024, 3)

data/goal_pos
  Shape : (345728, 3)
  Dtype : float32
  Chunks: (1024, 3)

data/goal_relative
  Shape : (345728, 3)
  Dtype : float32
  Chunks: (1024, 3)

data/point_cloud
  Shape : (345728, 1024, 3)
  Dtype : float32
  Chunks: (1024, 1024, 3)

data/point_valid_mask
  Shape : (345728, 1024)
  Dtype : bool
  Chunks: (1024, 1024)

data/robot_mask
  Shape : (345728, 1024)
  Dtype : bool
  Chunks: (1024, 1024)

data/sim_action
  Shape : (345728, 7)
  Dtype : float32
  Chunks: (1024, 7)

data/state
  Shape : (345728, 13)
  Dtype : float32
  Chunks: (1024, 13)

data/success
  Shape : (345728,)
  Dtype : bool
  Chunks: (1024,)

data/target_position
  Shape : (345728, 3)
  Dtype : float32
  Chunks: (1024, 3)

data/tcp_pose
  Shape : (345728, 7)
  Dtype : float32
  Chunks: (1024, 7)

data/trajectory_family_id
  Shape : (345728, 1

In [ ]:
import numpy as np

keep = {
    "data/action",
    "data/state",
    # "data/point_cloud",
    "data/target_position",
}

for name, arr in arrays:
    if name not in keep:
        continue

    print("=" * 80)
    print(name)
    print("=" * 80)

    data = arr[:]

    print("Shape:", data.shape)
    print("Dtype:", data.dtype)

    if data.ndim == 0:
        print(data)
    else:
        print("First 3 entries:")
        print(data[:5])

    print()

data/action
Shape: (345728, 7)
Dtype: float32
First 3 entries:
[[-0.11123215  0.1293497  -0.15033704  1.032148    0.01314411  0.9026994
  -0.26836753]
 [-0.10578173  0.10152289 -0.14820771  1.0037133   0.00745755  0.9017546
  -0.25791794]
 [-0.10051212  0.07344687 -0.1457196   0.9756275   0.00175524  0.9014277
  -0.24715434]
 [-0.09546019  0.04507544 -0.14279194  0.94796634 -0.00390156  0.90185004
  -0.23605877]
 [-0.09060843  0.01643753 -0.1394544   0.9207001  -0.00946207  0.90296555
  -0.22469752]]

data/state
Shape: (345728, 13)
Dtype: float32
First 3 entries:
[[-1.1123215e-01  1.2934969e-01 -1.5033704e-01  1.0321480e+00
   1.3144113e-02  9.0269941e-01 -2.6836753e-01  8.3999997e-01
   8.3999997e-01  8.3999997e-01  8.3999997e-01  8.3999997e-01
   8.3999997e-01]
 [-1.1123215e-01  1.2934969e-01 -1.5033704e-01  1.0321480e+00
   1.3144113e-02  9.0269941e-01 -2.6836753e-01  8.3999997e-01
   8.3999997e-01  8.3999997e-01  8.3999997e-01  8.3999997e-01
   8.3999997e-01]
 [-1.0386841e-01  1.07

In [7]:
import numpy as np

keep = {
    "data/action",
    "data/state",
    # "data/point_cloud",
    # "data/target_position",
}

for name, arr in arrays:
    if name not in keep:
        continue

    print("=" * 80)
    print(name)
    print("=" * 80)

    data = arr[:]

    print("Shape:", data.shape)
    print("Dtype:", data.dtype)

    if data.ndim == 0:
        print(data)
    else:
        # print("First 5 entries:")
        # print(data[:5])

        print("\nLast 5 entries:")
        print(data[-5:])

    print()

data/action
Shape: (345728, 7)
Dtype: float32

Last 5 entries:
[[ 0.40901858 -0.39462435  0.36459762  0.7935735   0.16964473  1.1625063
   0.66814494]
 [ 0.40901858 -0.39462435  0.36459762  0.7935735   0.16964473  1.1625063
   0.66814494]
 [ 0.40901858 -0.39462435  0.36459762  0.7935735   0.16964473  1.1625063
   0.66814494]
 [ 0.40901858 -0.39462435  0.36459762  0.7935735   0.16964473  1.1625063
   0.66814494]
 [ 0.40901858 -0.39462435  0.36459762  0.7935735   0.16964473  1.1625063
   0.66814494]]

data/state
Shape: (345728, 13)
Dtype: float32

Last 5 entries:
[[ 0.40992075 -0.39216176  0.36509275  0.78753126  0.20512486  1.1438285
   0.64932966  0.42251664  0.0364106   0.76981205  0.8399472   0.35702634
   0.8590283 ]
 [ 0.4107601  -0.39273486  0.36559704  0.7876303   0.20554274  1.1433758
   0.64905417  0.42085698  0.04412881  0.7597116   0.83995086  0.35913253
   0.859684  ]
 [ 0.40946683 -0.392442    0.3644844   0.78800505  0.20570305  1.1433232
   0.64927644  0.42233935  0.044297

In [6]:
import rerun as rr

rr.init("point_cloud", spawn=True)

2026-07-07T09:01:45.172385Z  INFO re_perf_telemetry::telemetry: Telemetry initialized enabled=false service=rerun trace_mode="off" traces=off logs=off metrics=off tracy=false
2026-07-07T09:01:45.184701Z  INFO winit::platform_impl::linux::x11::window: Guessed window scale factor: 1
2026-07-07T09:01:45.416424Z  WARN egui_wgpu::winit: Transparent window was requested, but the active wgpu surface does not support a `CompositeAlphaMode` with transparency.
2026-07-07T09:01:45.472943Z  INFO re_grpc_server: Listening for gRPC connections on 0.0.0.0:9876. Connect by running `rerun --connect rerun+http://127.0.0.1:9876/proxy`


In [7]:
import numpy as np
import rerun as rr
import zarr

root = zarr.open("../pg3d_xarm7_fixed.zarr", mode="r")

pc = root["data/point_cloud"]
episode_ends = root["meta/episode_ends"][:]

episode_id = 0  # change this

start = 0 if episode_id == 0 else episode_ends[episode_id - 1]
end = episode_ends[episode_id]

print(f"Episode {episode_id}: {start} -> {end} ({end-start} frames)")

rr.init("episode_view", spawn=True)

for t in range(start, end):
    pts = np.asarray(pc[t])

    colors = pts[:, 2]

    rr.log(
        "world/point_cloud",
        rr.Points3D(
            positions=pts,
            colors=colors,
            radii=0.003,
        ),
    )

Episode 0: 0 -> 73 (73 frames)


2026-07-07T09:02:12.360006Z ERROR re_grpc_client::write: Write messages call failed: transport error
